# 100 - MLflow + LlamaIndex: experimentos y RAG sin saturar

Este notebook añade dos herramientas al mix principal sin abrir demasiados frentes:

1. `MLflow` para registrar experimentos y comparar resultados.
2. `LlamaIndex` para entender RAG sobre documentos propios, usando `Chroma` o `FAISS` como idea interna de búsqueda vectorial.

Los ejemplos tienen fallback con Python estándar para que el notebook sea ejecutable aunque las librerías externas no estén instaladas.

## Preparación para Colab

La siguiente celda instala librerías opcionales solo si se detecta Google Colab. En local no instala nada automáticamente.

In [ ]:
import importlib.util
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
OPTIONAL_PACKAGES = {
    "mlflow": "mlflow",
    "llama_index": "llama-index",
    "chromadb": "chromadb",
    "faiss": "faiss-cpu",
}

def install_if_missing(import_name, package_name):
    if importlib.util.find_spec(import_name) is not None:
        print(f"OK: {package_name} ya está instalado")
        return
    print(f"Instalando {package_name}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

if IN_COLAB:
    for import_name, package_name in OPTIONAL_PACKAGES.items():
        install_if_missing(import_name, package_name)
else:
    print("No se ha detectado Colab. Instalación automática omitida.")
    print("Instalación manual opcional: pip install mlflow llama-index chromadb faiss-cpu")

## 1. MLflow: registrar experimentos

Primero comparamos varios modelos clásicos sobre Iris. La idea no es mejorar el modelo, sino registrar parámetros y métricas para poder justificar qué configuración funciona mejor.

In [ ]:
import json
from pathlib import Path

try:
    import mlflow
    HAS_MLFLOW = True
except Exception:
    mlflow = None
    HAS_MLFLOW = False

print("MLflow disponible:", HAS_MLFLOW)

Esta celda compara tres clasificadores sencillos implementados con Python estándar y calcula su accuracy. Es el tipo de experimento que tiene sentido registrar con MLflow, sin depender de scikit-learn para que el notebook sea más portable.

In [ ]:
from collections import defaultdict
from math import dist

dataset = [
    ([5.1, 3.5, 1.4, 0.2], "setosa"), ([4.9, 3.0, 1.4, 0.2], "setosa"),
    ([5.0, 3.4, 1.5, 0.2], "setosa"), ([4.6, 3.1, 1.5, 0.2], "setosa"),
    ([7.0, 3.2, 4.7, 1.4], "versicolor"), ([6.4, 3.2, 4.5, 1.5], "versicolor"),
    ([6.9, 3.1, 4.9, 1.5], "versicolor"), ([5.5, 2.3, 4.0, 1.3], "versicolor"),
    ([6.3, 3.3, 6.0, 2.5], "virginica"), ([5.8, 2.7, 5.1, 1.9], "virginica"),
    ([7.1, 3.0, 5.9, 2.1], "virginica"), ([6.5, 3.0, 5.8, 2.2], "virginica"),
]
train = dataset[:3] + dataset[4:7] + dataset[8:11]
test = [dataset[3], dataset[7], dataset[11]]

def majority_classifier(train_rows, sample):
    counts = defaultdict(int)
    for _, label in train_rows:
        counts[label] += 1
    return max(counts, key=counts.get)

def nearest_centroid_classifier(train_rows, sample):
    grouped = defaultdict(list)
    for features, label in train_rows:
        grouped[label].append(features)
    centroids = {
        label: [sum(values) / len(values) for values in zip(*rows)]
        for label, rows in grouped.items()
    }
    return min(centroids, key=lambda label: dist(sample, centroids[label]))

def one_nearest_neighbor_classifier(train_rows, sample):
    features, label = min(train_rows, key=lambda row: dist(sample, row[0]))
    return label

models = {
    "majority_baseline": majority_classifier,
    "nearest_centroid": nearest_centroid_classifier,
    "one_nearest_neighbor": one_nearest_neighbor_classifier,
}

results = []
for name, predict_fn in models.items():
    predictions = [predict_fn(train, features) for features, _ in test]
    expected = [label for _, label in test]
    accuracy = sum(p == y for p, y in zip(predictions, expected)) / len(expected)
    results.append({"model": name, "accuracy": round(accuracy, 4), "predictions": predictions})

print(json.dumps(results, indent=2, ensure_ascii=False))

Esta celda registra los resultados. Si MLflow está instalado, crea runs reales; si no, guarda un fichero JSON equivalente para mantener el ejemplo ejecutable.

In [ ]:
def log_experiments(results):
    if HAS_MLFLOW:
        mlflow.set_experiment("semana27_iris_tracking")
        for row in results:
            with mlflow.start_run(run_name=row["model"]):
                mlflow.log_param("model", row["model"])
                mlflow.log_metric("accuracy", row["accuracy"])
        return {"backend": "mlflow", "runs": len(results)}

    fallback = {"experiment": "semana27_iris_tracking", "runs": results}
    Path("mlflow_fallback_iris.json").write_text(
        json.dumps(fallback, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return {"backend": "json_fallback", "file": "mlflow_fallback_iris.json"}

tracking_result = log_experiments(results)
best = max(results, key=lambda row: row["accuracy"])
print(tracking_result)
print("Mejor modelo:", best)

## 2. LlamaIndex: RAG con documentos propios

Ahora simulamos el flujo central de LlamaIndex: documentos, recuperación de contexto y respuesta fundamentada. `Chroma` o `FAISS` serían la pieza interna encargada de buscar vectores parecidos.

In [ ]:
from collections import Counter
from math import sqrt
import re

DOCS = [
    {"id": "d1", "title": "MLflow", "text": "MLflow registra experimentos, parámetros, métricas y artefactos."},
    {"id": "d2", "title": "LlamaIndex", "text": "LlamaIndex facilita construir sistemas RAG sobre documentos propios."},
    {"id": "d3", "title": "Chroma", "text": "Chroma es una base vectorial cómoda para prototipos de recuperación semántica."},
    {"id": "d4", "title": "FAISS", "text": "FAISS permite búsqueda vectorial local eficiente por similitud."},
]

def tokenize(text):
    return re.findall(r"[a-záéíóúñü]+", text.lower())

def cosine(a, b):
    common = set(a) & set(b)
    numerator = sum(a[token] * b[token] for token in common)
    norm_a = sqrt(sum(value * value for value in a.values()))
    norm_b = sqrt(sum(value * value for value in b.values()))
    return numerator / (norm_a * norm_b) if norm_a and norm_b else 0.0

print(f"Documentos disponibles: {len(DOCS)}")

Esta celda implementa un índice vectorial mínimo. No sustituye a Chroma ni FAISS, pero permite entender su papel: guardar representaciones de documentos y recuperar los más similares a una consulta.

In [ ]:
class MiniVectorIndex:
    def __init__(self, documents, backend="mini-faiss"):
        self.documents = documents
        self.backend = backend
        self.vectors = [Counter(tokenize(doc["title"] + " " + doc["text"])) for doc in documents]

    def retrieve(self, question, top_k=2):
        query_vector = Counter(tokenize(question))
        scored = []
        for doc, vector in zip(self.documents, self.vectors):
            scored.append((cosine(query_vector, vector), doc))
        scored.sort(key=lambda item: item[0], reverse=True)
        return [{"score": round(score, 3), **doc} for score, doc in scored[:top_k] if score > 0]

index = MiniVectorIndex(DOCS, backend="mini-chroma-faiss")
question = "Qué herramienta uso para buscar documentos parecidos en un RAG?"
contexts = index.retrieve(question, top_k=2)
print(json.dumps(contexts, indent=2, ensure_ascii=False))

Esta celda genera una respuesta simulada usando solo los documentos recuperados. Es la parte que en una aplicación LlamaIndex real conectaría con un modelo de lenguaje.

In [ ]:
def rag_answer(question, contexts):
    if not contexts:
        return "No hay contexto suficiente para responder."
    evidence = " ".join(ctx["text"] for ctx in contexts)
    sources = ", ".join(ctx["title"] for ctx in contexts)
    return f"Pregunta: {question}\nRespuesta simulada: {evidence}\nFuentes: {sources}"

answer = rag_answer(question, contexts)
print(answer)

## 3. Registrar también el experimento RAG

MLflow no solo sirve para modelos clásicos. También puede registrar configuraciones de RAG: backend vectorial, `top_k`, número de documentos recuperados o métricas simples.

In [ ]:
def evaluate_context(question, contexts):
    q_terms = set(tokenize(question))
    c_terms = set(tokenize(" ".join(ctx["text"] for ctx in contexts)))
    return round(len(q_terms & c_terms) / max(1, len(q_terms)), 3)

rag_metrics = {
    "num_contexts": len(contexts),
    "context_relevance_proxy": evaluate_context(question, contexts),
}
rag_params = {"vector_backend": index.backend, "top_k": 2, "framework_style": "llamaindex"}

if HAS_MLFLOW:
    mlflow.set_experiment("semana27_rag_tracking")
    with mlflow.start_run(run_name="mini_llamaindex_rag"):
        for key, value in rag_params.items():
            mlflow.log_param(key, value)
        for key, value in rag_metrics.items():
            mlflow.log_metric(key, value)
else:
    rag_run = {"params": rag_params, "metrics": rag_metrics, "question": question, "answer": answer}
    Path("mlflow_fallback_rag.json").write_text(
        json.dumps(rag_run, indent=2, ensure_ascii=False), encoding="utf-8"
    )

print({"params": rag_params, "metrics": rag_metrics})

## Resumen

1. `MLflow` aporta trazabilidad: qué probamos, con qué parámetros y qué resultado dio.
2. `LlamaIndex` aporta estructura para RAG sobre documentos.
3. `Chroma` y `FAISS` se entienden aquí como motores internos de recuperación vectorial.
4. No se añade MCP como herramienta nueva porque ya aparece en el notebook 98 y puede haberse trabajado en otro módulo.

## Reto adicional

1. Añade otro clasificador y registra su accuracy.
2. Añade documentos nuevos al RAG.
3. Prueba `top_k=1`, `top_k=2` y `top_k=3`.
4. Registra cada configuración como una ejecución distinta y compara los resultados.

## Para profundizar
Este notebook introduce MLflow + LlamaIndex. La documentación completa incluye:
- Model Registry, Serving, Vector Stores

Consulta el documento correspondiente en Documentacion/.

In [ ]:
# Los ejemplos avanzados están en los documentos de Documentacion/# Consulta el archivo correspondiente según lo que quieras profundizar.